# 2026-03-20 (금) RAG 파이프라인 완성 - FAISS + LCEL + RunnableParallel

## W2 Day 4: 벡터스토어 저장/로드, LCEL 체인, Source 추적까지

### 오늘 배우는 것

어제 만든 DocumentStore는 **매번 임베딩을 다시 계산**하는 비효율적인 구조였다.
오늘은 **FAISS** 벡터스토어로 한번 계산해 저장해두고 반복 사용하는 방식을 배운다.
또한 **LCEL (LangChain Expression Language)**의 파이프 문법으로 RAG 파이프라인을 우아하게 연결한다.

1. 문서를 섹션 단위로 쪼개 Document 리스트 만들기
2. **FAISS** 벡터스토어 생성, 저장, 로드
3. **Retriever** 인터페이스로 유사 문서 검색
4. **LCEL**: `prompt | llm | parser` 파이프 체인
5. **RunnablePassthrough / RunnableParallel**로 question + context 동시 주입
6. 답변과 함께 **Source Document**까지 반환
7. **Streaming** 응답 + threshold로 **out-of-scope 질문 차단**

### 비유로 이해하는 FAISS + LCEL

> **어제의 DocumentStore**: 매번 도서관 가서 **책 1권씩 임베딩 작업을 반복**하는 사람
> **오늘의 FAISS**: 책을 한 번 임베딩해 **라벨 붙여 서가에 꽂아두고**, 질문 오면 **서가에서 바로 뽑는** 사서
>
> **LCEL 파이프(`|`)**: 리눅스 파이프처럼 **앞 결과를 다음 단계로 넘기는 컨베이어 벨트**
> `질문 | 검색기 | 포매터 | 프롬프트 | LLM | 파서` → 한 줄로 RAG 완성

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260320_RAG_Document.ipynb)

## 0. Colab 환경 세팅

FAISS는 Facebook(Meta)이 만든 벡터 유사도 검색 라이브러리.
`faiss-cpu`를 추가 설치한다.

In [ ]:
# Colab 환경 전용 설치 (로컬 실행 시 주석 처리)
!pip install -q langchain-text-splitters
!pip install -q langchain-openai
!pip install -q langchain_classic
!pip install -q langchain-community
!pip install -q faiss-cpu
!pip install -q python-dotenv

## 1. 라이브러리 import & API 키 세팅

오늘 새로 등장하는 친구들:
- `FAISS`: 벡터스토어 (`langchain_community.vectorstores`)
- `ChatPromptTemplate`: 메시지 템플릿 (system/user 역할별)
- `StrOutputParser`: LLM의 AIMessage에서 `.content` 문자열만 뽑는 파서
- `RunnablePassthrough`: **입력을 그대로 통과**시키는 Runnable (복수 key에 동일 입력을 꽂을 때 유용)
- `RunnableParallel`: 여러 체인을 **병렬 실행**해 딕셔너리로 모으기
- `RunnableBranch`, `RunnableLambda`: 조건 분기, 일반 함수를 체인에 붙이기

In [ ]:
import os
import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

# LangChain 핵심
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 오늘의 신규 - LCEL 필수 재료
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS  # 벡터스토어
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableBranch, RunnableLambda

from sklearn.metrics.pairwise import cosine_similarity

# Colab에서 API 키 가져오기
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
# LLM + Embedding 모델 초기화
# 동일 쿼리에 대해 임베딩 값은 항상 같으므로, 한번 계산해서 저장해두면 재사용 가능
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)

## 2. 샘플 데이터 재생성 (어제 작업물 재사용)

세션이 끊겨도 바로 실행할 수 있도록 `sample_data/` 폴더를 다시 만든다.
어제와 같은 3종 문서(사내규정/AI보고서/제품매뉴얼).

In [ ]:
# sample_data 폴더와 파일 재생성 (Colab 런타임 리셋 대비)
data_dir = Path('sample_data')
data_dir.mkdir(exist_ok=True)

sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.

제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

1. 개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

2. 주요 트렌드
- RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
- 멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
- AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
- 소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

3. 시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

1. 제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

2. 초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

3. 주요 기능
- 음성 명령: \"허브야, 거실 조명 켜줘\" 등의 자연어 명령 지원
- 자동 스케줄: 시간대별 기기 자동 제어
- 에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
- 보안 모드: 외출 시 자동 보안 설정
"""
}

for filename, content in sample_texts.items():
    (data_dir / filename).write_text(content, encoding='utf-8')

## 3. 문서를 섹션 단위로 쪼개 Document 만들기

**단위 선택이 중요!** 파일 통째로 Document 하나로 묶으면 검색 결과가 뭉툭해진다.
오늘은 **빈 줄(`\n\n`) 기준**으로 섹션을 나눠 **13개 Document**를 만들 예정.

### 비유

> 책을 통째로 하나의 Document로 넣으면 → **"이 책 어딘가에 있어요"**
> 장(chapter) 단위로 쪼개면 → **"3장 45페이지쯤"** - 훨씬 정확

### 메타데이터 활용

- `category`: 문서 카테고리(사내규정/제품매뉴얼/AI보고서) - 검색 후 어디서 온 정보인지 알려줌
- `source`: 원본 파일명

In [ ]:
# 파일마다 카테고리 라벨을 부여한 딕셔너리
files = {
    data_dir / 'company_policy.txt': '사내규정',
    data_dir / 'product_manual.txt': '제품매뉴얼',
    data_dir / 'ai_report.txt': 'AI보고서'
}

documents = []
# 각 파일을 열어 빈 줄(\n\n) 기준으로 섹션을 분리
for fpath, category in files.items():
    text = fpath.read_text(encoding='utf-8')
    # 두 줄 개행 = 섹션 구분자
    for section in text.strip().split('\n\n'):
        if section.strip():  # 빈 섹션 스킵
            documents.append(Document(
                page_content=section.strip(),
                metadata={'category': category, 'source': fpath.name}
            ))

In [ ]:
# 총 몇 개 섹션으로 나뉘었는지 확인 (예: 13개)
len(documents)

In [ ]:
# 첫 번째 Document 확인 - 메타데이터와 본문 구조 확인
documents[0]

## 4. FAISS 벡터스토어 만들기

`FAISS.from_documents(documents, embeddings)`:
- 내부에서 각 Document의 `page_content`를 **embeddings 모델로 벡터화**
- 벡터를 FAISS **인덱스**에 저장 (빠른 최근접 검색 구조)
- 원본 Document도 함께 보관 (검색 시 같이 반환)

### FAISS가 빠른 이유

- 단순 코사인 유사도는 O(N×D) — 문서 많을수록 느려짐
- FAISS는 **근사 최근접 이웃(ANN)** 알고리즘으로 **O(log N)**에 가깝게 검색
- 우리 실습은 13개라 차이가 안 느껴지지만, 수십만 건에선 압도적

In [ ]:
# Document 리스트 → 벡터스토어로 변환 (임베딩 자동 실행됨)
vectorstore = FAISS.from_documents(documents, embeddings)

## 5. Retriever - 검색 인터페이스

`vectorstore.as_retriever()`는 벡터스토어를 **검색 전용 객체**로 감싼다.
Retriever는 LangChain 체인에서 **표준 검색 모듈**로 동작하며, `invoke(query)`로 호출.

### search_kwargs 옵션
- `k`: 상위 몇 개 문서 반환할지 (기본 4)
- `score_threshold`: 유사도 임계값 설정
- `fetch_k`, `lambda_mult`: MMR (다양성 검색) 옵션

In [ ]:
# Retriever 생성 - 상위 3개 문서 반환
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

## 6. 벡터스토어 저장 & 로드

**임베딩 호출은 돈이 든다!** 같은 문서를 매번 임베딩하는 건 낭비.
→ 한번 만든 벡터스토어를 파일로 저장해두고 재사용한다.

### 저장 포맷
- `save_local('폴더명')` → `폴더명/index.faiss`, `폴더명/index.pkl` 생성
- `index.pkl`은 **pickle 파일** → 악성 코드 주입 위험이 있어 로드 시 명시적 허용 필요

### allow_dangerous_deserialization=True
- "내가 만든 파일이니 신뢰한다"는 의미
- 외부에서 받은 파일이라면 절대 True로 두면 안 됨

In [ ]:
# 벡터스토어를 로컬 폴더에 저장
vectorstore.save_local('faiss_docs')

In [ ]:
# 저장한 벡터스토어 로드 - embeddings 객체는 동일한 걸 써야 함
loaded_vs = FAISS.load_local(
    'faiss_docs',
    embeddings,
    allow_dangerous_deserialization=True  # pickle 로드 허용
)

In [ ]:
# 저장/로드가 잘 됐는지 확인 - 총 벡터 개수 (13개여야 함)
loaded_vs.index.ntotal

In [ ]:
# 실제 검색 테스트 - '재택근무 규정'으로 상위 3개 문서 반환
retriever.invoke('재택근무 규정')

## 7. LCEL (LangChain Expression Language)

`|` 파이프 연산자로 **체인을 우아하게 연결**하는 문법.
리눅스의 `cat foo.txt | grep bar | wc -l`처럼, 앞 단계의 출력이 다음 단계의 입력이 된다.

### 기본 체인 구조

```
prompt | llm | StrOutputParser()
```

- `prompt`: 템플릿에 변수 주입 → ChatPromptValue
- `llm`: 프롬프트를 받아 AI 응답 생성 → AIMessage
- `StrOutputParser()`: AIMessage.content 문자열만 추출

### 왜 이렇게 쓰는가?

- **가독성**: 데이터 흐름이 한눈에 보임
- **재사용성**: 중간 단계 교체가 쉬움
- **배치/스트리밍**: invoke/batch/stream을 동일 체인에 적용 가능

In [ ]:
# LCEL : LangChain Expression Language
# prompt | llm | StrOutputParser()

# 간단한 체인: 질문을 받아 그대로 LLM에 전달
simple_prompt = ChatPromptTemplate.from_messages([
    ('user', '{question}')  # user 역할 메시지에 question 변수 주입
])

# 파이프로 연결 - 순서: 프롬프트 → LLM → 문자열 추출
simple_chain = simple_prompt | llm | StrOutputParser()

In [ ]:
# 체인 실행 - invoke 함수에 프롬프트 변수를 딕셔너리로 전달
result = simple_chain.invoke({'question': '대한민국의 수도는?'})

In [ ]:
print(result)

## 8. RunnablePassthrough - 입력을 그대로 흘려보내기

RAG 체인에서는 **사용자 질문을 두 군데로 보내야** 한다:
1. **검색용** → retriever로 들어가 context 생성
2. **프롬프트 원본** → LLM에게 원래 질문 전달

`RunnablePassthrough()`는 입력을 **그대로 복사해 다음 단계에 넘겨주는** 역할.

> 비유: **Y자형 파이프** - 들어온 물을 두 갈래로 나눠 보내는 분배기

In [ ]:
# RunnablePassthrough 동작 확인
passthrough = RunnablePassthrough()

In [ ]:
# 문자열도
passthrough.invoke('안녕하세요')

In [ ]:
# 숫자도
passthrough.invoke(123)

In [ ]:
# 딕셔너리도 그대로 통과
passthrough.invoke({'a': 1})

## 9. 포맷 함수 - retriever 결과를 프롬프트용 문자열로

Retriever는 `List[Document]`를 반환하는데, LLM 프롬프트에는 **텍스트**가 들어가야 한다.
→ 중간에 **포매팅 함수** 하나를 끼워넣는다.

### 포인트: 일반 파이썬 함수도 LCEL 파이프에 그대로 사용 가능!
LangChain은 함수를 자동으로 `RunnableLambda`로 감싸준다.

In [ ]:
# Document 리스트를 프롬프트용 문자열로 변환
def format_docs(docs):
    # 카테고리 라벨 + 본문을 '--'로 구분해 이어붙이기
    return '\n--\n'.join(
        f"[{d.metadata.get('category', '')}] {d.page_content}"
        for d in docs
    )

In [ ]:
# 동작 확인 - 먼저 검색만
docs = retriever.invoke('재택근무')

In [ ]:
docs  # List[Document] 형태

In [ ]:
# 포맷 함수로 하나의 문자열로 변환
formatted = format_docs(docs)

In [ ]:
print(formatted)

In [ ]:
# retriever와 format_docs를 파이프로 연결 - 재사용 가능한 작은 체인
retriever_chain = retriever | format_docs
context_text = retriever_chain.invoke('재택근무')

In [ ]:
print(context_text)

## 10. RAG Chain 완성형

### 구조
```
{
  "context": retriever | format_docs,
  "question": RunnablePassthrough()
} | rag_prompt | llm | StrOutputParser()
```

### 해석
- 딕셔너리가 나와 context와 question을 **동시에 만들기** (암묵적 RunnableParallel)
- `context`: 질문을 retriever로 보내 검색 → format_docs로 문자열화
- `question`: 질문을 그대로 통과시켜 원문 유지
- 그걸 프롬프트 템플릿에 주입 → LLM → 문자열 파싱

### 비유

> Y자 파이프로 분기 → 한쪽은 **검색 결과**로, 한쪽은 **원질문 그대로**로
> → 프롬프트 조립 기계에 둘 다 투입 → 완성된 프롬프트가 LLM으로

In [ ]:
# RAG 프롬프트 템플릿 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', '사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요.'),
    ('user', '{question}')
])

# RAG 체인 - 딕셔너리 자리에 retriever 체인과 RunnablePassthrough를 동시에 배치
rag_chain = (
    {
        'context': retriever | format_docs,      # 검색 결과를 문자열로
        'question': RunnablePassthrough()         # 질문 그대로 통과
    }
    | rag_prompt       # 프롬프트 템플릿 주입
    | llm              # LLM 호출
    | StrOutputParser()  # 문자열 추출
)

### 실습: k=1로 바꾼 새 체인과 비교

검색 결과를 **1개만** 가져오는 체인을 따로 만들어 성능을 비교해보자.
문서가 적을 때 k=1이면 답변이 짧고 정확하지만, 맥락이 부족해 틀릴 수 있다.

In [ ]:
# k=1 버전의 retriever
retriever_k1 = vectorstore.as_retriever(search_kwargs={'k': 1})

# 동일한 rag_prompt/llm/parser를 재활용 (retriever만 교체)
rag_chain_k1 = (
    {
        'context': retriever_k1 | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# k=1 체인으로 답변 생성
answer = rag_chain_k1.invoke('재택근무 규정이 어떻게 되나요?')
print(answer)

In [ ]:
# 여러 질문 일괄 테스트 - 각기 다른 카테고리 문서에서 답변이 나오는지 확인
question_list = [
    '스마트홈 허브 초기 설정 방법',  # 제품매뉴얼에서
    'ai 산업 성장률',                # AI보고서에서
    '교육비 지원 한도'                # 사내규정에서
]

for q in question_list:
    print(f"Q : {q}")
    print(f"A : {rag_chain_k1.invoke(q)}")
    print("======")

## 11. RunnableParallel - 답변과 출처를 동시에 반환

답변만 받으면 **근거를 알 수 없다**. "AI가 그냥 지어낸 건 아닐까?" 의심이 남음.
→ 답변과 검색된 문서(source documents)를 **병렬로** 반환하자.

`RunnableParallel`:
- 여러 체인을 **동시에 실행**
- 결과를 딕셔너리로 모아줌

> 비유: **Y자 분기를 한번 더** - 같은 질문을 두 갈래로 보내 한쪽은 답변, 다른 쪽은 원본 문서를 받아 묶음 포장

In [ ]:
# 프롬프트에 '참고 문서에 없으면 해당 정보 없음' 안전장치 추가
rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요.'
     "참고 문서에 없는 내용이면, '해당 정보가 없습니다' 라고 답변하세요."),
    ('user', '{question}')
])

# 다시 RAG 체인 (새 프롬프트 적용)
rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke('재택근무 규정이 어떻게 되나요?')

In [ ]:
# RunnableParallel로 답변 + 원본 문서를 함께 반환
rag_chain_with_sources = RunnableParallel(
    answer=rag_chain,           # 최종 답변
    source_document=retriever   # 출처 문서 리스트
)

In [ ]:
# 결과는 딕셔너리 {'answer': ..., 'source_document': [Document, ...]}
rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')

In [ ]:
# source 문서만 깔끔하게 출력
result = rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')
for doc in result['source_document']:
    print(f" [{doc.metadata['category']}] {doc.page_content[:100]}")

## 12. 스트리밍 응답

ChatGPT처럼 **답변을 조금씩 실시간으로** 받고 싶을 때 `stream()` 메서드 사용.
체인 구조는 똑같고, `invoke` 대신 `stream`만 호출하면 됨.

사용자 경험(UX)이 훨씬 좋아짐 → 긴 답변도 대기 없이 읽기 시작 가능.

In [ ]:
# 스트리밍 - 청크 단위로 실시간 출력
for chunk in rag_chain.stream('재택근무 규정이 어떻게 되나요?'):
    print(chunk, end='', flush=True)  # end='' 와 flush로 이어붙이기

## 13. Out-of-scope 질문 차단 - RunnableBranch로 조건 분기

RAG의 큰 문제: 문서에 없는 질문에도 **LLM이 자기 지식으로 답해버림** → Hallucination 위험.
예) "2002년 월드컵 우승팀은?" → 문서에 없는데도 답변 나올 수 있음.

### 해결
- 검색 결과의 **유사도 점수를 확인**
- 임계값(threshold)을 넘지 못하면 **"해당 정보가 없습니다"** 반환
- 그 외에는 정상 RAG 체인 실행

### 사용 함수
- `similarity_search_with_score(query, k)`: 점수까지 같이 반환
- `RunnableBranch((조건, 분기1), 기본분기)`: if-else 체인 분기
- `RunnableLambda(func)`: 일반 함수를 체인에 넣기

In [ ]:
# 문서 밖 질문 예시 - RAG는 '해당 정보가 없습니다' 답변해야 이상적
rag_chain.invoke('2002년 월드컵 우승팀은 어디인가요?')

In [ ]:
# 점수 기반 사전 검증 함수
def check_and_prepare(question):
    # similarity_search_with_score: (Document, 거리점수) 튜플 리스트 반환
    # 주의: FAISS의 기본 거리는 L2 - '작을수록' 유사
    results = vectorstore.similarity_search_with_score(question, k=3)
    docs = [doc for doc, _ in results]
    return {
        'question': question,
        'context': format_docs(docs),
        'score': results[0][1]  # top-1의 거리 점수
    }

# safe_chain: 점수가 너무 높으면(유사도 낮으면) 정중히 거절
safe_chain = (
    RunnableLambda(check_and_prepare)  # 일반 함수를 체인에 붙이기
    | RunnableBranch(
        # 조건 튜플: (조건 함수, 해당 조건일 때 실행할 함수)
        (lambda x: x['score'] > 1.5, lambda x: '해당 정보가 없습니다'),
        # 마지막 인자 = 기본 분기 (else)
        rag_prompt | llm | StrOutputParser()
    )
)

In [ ]:
# 문서에 없는 질문 → 차단됨
safe_chain.invoke('2002년 월드컵 우승팀은 어디인가요?')

In [ ]:
# 문서에 있는 질문 → 정상 답변
safe_chain.invoke('재택근무 규정을 알려주세요?')

## 14. 종합 실습 - answer + source + context 한 번에

RunnableParallel로 **답변 / 원본 문서 / 포맷된 컨텍스트**를 한꺼번에 반환.
디버깅용으로도 좋고, 실제 서비스에서 **"근거 보기" 버튼**에 연결하기도 좋다.

In [ ]:
# answer, source_documents, 포맷된 context를 동시에 리턴
new_chain = RunnableParallel(
    answer=rag_chain,
    source_documents=retriever,
    context=retriever | format_docs  # 포맷 파이프 재사용
)
result = new_chain.invoke('재택근무 규정을 알려주세요')

In [ ]:
result  # 세 키가 모두 들어있는 딕셔너리

## 15. 점수 기반 검증 로직 직접 만들기

`verify_query` 함수: 쿼리를 받아 **threshold 이하일 때만 문서 내용을 반환**, 아니면 '해당 정보 없음'.
체인 내부에서 쓰지 않고 독립 함수로 테스트해볼 때 편리.

In [ ]:
# 리트리버 동작 확인 - 문서 밖 질문도 '가장 비슷한 것'은 뽑아온다 (점수 무관)
retriever.invoke('2002년 월드컵')

In [ ]:
# 점수까지 확인 - FAISS의 L2 거리는 작을수록 유사
vectorstore.similarity_search_with_score('재택근무 규정', k=3)

In [ ]:
# 문서+점수 예쁘게 출력
for doc, score in vectorstore.similarity_search_with_score('재택근무 규정', k=3):
    print(f" [{doc.metadata['category']}] {doc.page_content}")

In [ ]:
# 여러 쿼리로 top-1 카테고리/점수 비교
test_queries = [
    {'query': '재택근무 몇 회?'},
    {'query': '스마트홈 초기 설정'},
    {'query': 'RAG 기술 동향'},
]

for tq in test_queries:
    results = vectorstore.similarity_search_with_score(tq['query'], k=1)
    top_cat = results[0][0].metadata['category']
    context = results[0][0].page_content
    top_score = results[0][1]
    print(f"{tq} : top category : {top_cat}, {top_score}, {context}")

In [ ]:
# threshold를 넘기면 out-of-scope 로 판정하는 함수
def verify_query(query, threshold=1.3, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)

    # 최저 거리(= 최고 유사도) 문서를 찾기
    best_doc = None
    best_score = float('inf')  # 무한대로 초기화

    for doc, score in results:
        if score < best_score:
            best_score = score
            best_doc = doc

    # 가장 가까운 문서조차 threshold를 넘으면 out-of-scope
    if best_score > threshold:
        return '해당 정보가 없습니다'

    return best_doc.page_content

In [ ]:
# 문서 밖 질문
verify_query('2002년 월드컵')

## 16. 정리

### 오늘 얻은 것
1. **FAISS 벡터스토어** → 임베딩을 파일로 저장/로드, 빠른 검색
2. **Retriever** → 표준 검색 인터페이스, `k` 조정으로 검색 폭 제어
3. **LCEL 파이프** → `prompt | llm | parser` 깔끔한 체인
4. **RunnablePassthrough** → 입력 분기를 위한 Y자 파이프
5. **RunnableParallel** → 답변 + 출처 동시 반환
6. **RunnableBranch + threshold** → out-of-scope 질문 차단
7. **stream()** → 실시간 스트리밍 응답

### 다음 단계 예고 (w3)
- **Text Splitter** 심화 (RecursiveCharacterTextSplitter의 separators 튜닝)
- **Memory** - 대화 히스토리를 유지하는 챗봇
- **Agent + Tool** - RAG를 한 도구로, 더 큰 에이전트로 확장

## Mina 메모

- `vectorstore.as_retriever()` 오타 주의 (`as_retrever`로 쓰면 AttributeError)
- FAISS 로드 시 `allow_dangerous_deserialization=True` 옵션 까먹기 쉬움
- `RunnablePassthrough()`는 괄호 필수 - 클래스 호출해서 인스턴스를 만들어야 함
- RAG 체인의 딕셔너리 `{'context': ..., 'question': ...}`은 암묵적으로 RunnableParallel로 변환됨
- threshold 값(1.5)은 실험으로 튜닝해야 함 - L2 거리 기준이라 0에 가까울수록 유사
- 스트리밍은 `print(chunk, end='', flush=True)` 삼종 세트 꼭 기억